In [ ]:
import os
import re
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import shutil
import tensorflow as tf
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# dataset path
dataset_path = "/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset"

In [ ]:
# Function to count the number of images in each folder
def count_images_in_folders(base_path, folder_list):
    counts = {}
    for folder in folder_list:
        folder_path = os.path.join(base_path, folder)
        if os.path.exists(folder_path):
            counts[folder] = len(os.listdir(folder_path)) 
        else:
            counts[folder] = 0
    return counts

folders = ["train/benign", "train/malignant", "test/benign", "test/malignant"]
image_counts = count_images_in_folders(dataset_path, folders)

# to print the count of images
for folder, count in image_counts.items():
    print(f"{folder}: {count} images")

In [ ]:
# paths for train and test folders
train_benign_path = os.path.join(dataset_path, "train", "benign")
train_malignant_path = os.path.join(dataset_path, "train", "malignant")
test_benign_path = os.path.join(dataset_path, "test", "benign")
test_malignant_path = os.path.join(dataset_path, "test", "malignant")

train_path = '/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset/train'
validation_path = '/kaggle/working/validation'

In [ ]:
# Function to check filename consistency
def check_filename_consistency(base_path, subfolders):
    for subfolder in subfolders:
        folder_path = os.path.join(base_path, subfolder)
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path): 
                if not re.match(r"melanoma_\d+\.jpg", filename):
                    print(f"Issue in {subfolder}: {filename}")

check_filename_consistency(dataset_path, folders)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image_sizes = []
train_path = '/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset/train'
for class_name in os.listdir(train_path):
    class_path = os.path.join(train_path, class_name)
    for img_name in os.listdir(class_path)[:100]:  # Analyze first 100 images
        img = cv2.imread(os.path.join(class_path, img_name))
        image_sizes.append(img.shape[:2])  # Height, Width

image_sizes = np.array(image_sizes)
plt.scatter(image_sizes[:,1], image_sizes[:,0], alpha=0.5)
plt.xlabel("Width")
plt.ylabel("Height")
plt.title("Image Size Distribution")
plt.show()


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array

def compute_mean_std(directory):
    images = []
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        for img_name in os.listdir(class_path)[:100]:  # Sample first 100 images
            img = load_img(os.path.join(class_path, img_name))
            img_array = img_to_array(img) / 255.0  # Normalize
            images.append(img_array)
    images = np.array(images)
    mean = np.mean(images, axis=(0, 1, 2))
    std = np.std(images, axis=(0, 1, 2))
    return mean, std

mean, std = compute_mean_std(train_path)
print(f"Mean: {mean}, Standard Deviation: {std}")


In [ ]:
 #Deletes the validation folder
os.makedirs(validation_path, exist_ok=True)  # Recreate it fresh

def create_validation_split(train_path, validation_base_path, categories, split_ratio=0.2):
    for category in categories:
        source_folder = os.path.join(train_path, category)
        if not os.path.exists(source_folder):
            print(f"⚠️ Source folder {source_folder} does not exist. Skipping...")
            continue

        target_folder = os.path.join(validation_base_path, category)
        os.makedirs(target_folder, exist_ok=True)

        images = os.listdir(source_folder)
        train_images, val_images = train_test_split(images, test_size=split_ratio, random_state=42)

        for val_image in val_images:
            src = os.path.join(source_folder, val_image)
            dst = os.path.join(target_folder, val_image)
            if not os.path.exists(dst):  # Avoid duplicates
                shutil.copy2(src, dst)

    print("✅ Validation split created successfully!")
train_path = '/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset/train'
create_validation_split(train_path, validation_path, ["benign", "malignant"])


In [ ]:
import os

# Collect all filenames from train and validation sets
train_files = set()
val_files = set()

for category in ["benign", "malignant"]:
    train_files.update(os.listdir(os.path.join(train_path, category)))
    val_files.update(os.listdir(os.path.join(validation_path, category)))

# Find common files
common_files = train_files.intersection(val_files)

print("⚠️ Remaining duplicate images:", list(common_files))


for file in common_files:
    for category in ["benign", "malignant"]:
        val_file_path = os.path.join(validation_path, category, file)
        if os.path.exists(val_file_path):
            os.remove(val_file_path)
            print(f"🗑️ Deleted {file} from validation set!")



train_files = set()
val_files = set()

for category in ["benign", "malignant"]:
    train_files.update(os.listdir(os.path.join(train_path, category)))
    val_files.update(os.listdir(os.path.join(validation_path, category)))

# Check again for duplicates
common_files = train_files.intersection(val_files)

if len(common_files) == 0:
    print("✅ No duplicate images left! Training and validation sets are fully cleaned.")
else:
    print(f"⚠️ {len(common_files)} duplicate images still exist. Manual check needed!")


In [ ]:
# Counting abd printing images in train/test directories
train_benign = len(os.listdir(train_benign_path))
train_malignant = len(os.listdir(train_malignant_path))
test_benign = len(os.listdir(test_benign_path))
test_malignant = len(os.listdir(test_malignant_path))

print(f"train_benign: {train_benign} images")
print(f"train_malignant: {train_malignant} images")
print(f"test_benign: {test_benign} images")
print(f"test_malignant: {test_malignant} images")

In [ ]:
# Dictionary for visualization
class_counts = {
    "Train Benign": train_benign,
    "Train Malignant": train_malignant,
    "Test Benign": test_benign,
    "Test Malignant": test_malignant
}

# Bar Plot using Seaborn
plt.figure(figsize=(8, 6))
sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()), palette='coolwarm')
plt.xlabel('Class', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.title('Class Distribution in Dataset', fontsize=14)
plt.xticks(rotation=45)
plt.show()

# Bar Plot using Matplotlib
plt.bar(class_counts.keys(), class_counts.values(), color=['blue', 'red', 'blue', 'red'])
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.title('Class Distribution in Dataset')
plt.xticks(rotation=45) 
plt.show()

# Train-test ratio (v-imp)
train_ratio = train_benign / train_malignant
test_ratio = test_benign / test_malignant

print(f"Train Benign to Malignant Ratio: {train_ratio:.2f}")
print(f"Test Benign to Malignant Ratio: {test_ratio:.2f}")

In [ ]:
# basic augmentation using imagedatagenerator function
train_datagen = ImageDataGenerator(
    rescale=1./255,
    featurewise_center=True,  
    featurewise_std_normalization=True,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1
)


In [ ]:
'''train_dir = "/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/"
print(os.listdir(train_dir))


# function to take augmented images from generator and visualize them
def show_augmented_images(generator, num_images=5):
    images, labels = next(generator) 
    
    plt.figure(figsize=(12, 6))
    for i in range(num_images):
        plt.subplot(1, num_images, i + 1)
        plt.imshow(images[i])  
        plt.axis("off")  
    plt.show()

show_augmented_images(train_generator)

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define augmentation parameters
train_datagen = ImageDataGenerator(
    rescale=1./255,           # Normalize pixel values
    rotation_range=20,         # Randomly rotate images up to 20 degrees
    width_shift_range=0.1,     # Shift width by 10%
    height_shift_range=0.1,    # Shift height by 10%
    zoom_range=0.1,            # Random zoom within 10%
    horizontal_flip=True,      # Flip horizontally
    brightness_range=[0.8,1.2] # Adjust brightness
)

# No augmentation for validation/test set
val_datagen = ImageDataGenerator(rescale=1./255)

# Load datasets
train_generator = train_datagen.flow_from_directory(
    'your_data/train', 
    target_size=(224, 224),  # Resize images for CNN
    batch_size=32,
    class_mode='binary'  # Change to 'categorical' if more than 2 classes
)

validation_generator = val_datagen.flow_from_directory(
    'your_data/validation', 
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

from tensorflow.keras.preprocessing.image import ImageDataGenerator
'''
datagen = ImageDataGenerator(
    rescale=1./255, 
    featurewise_center=True, 
    featurewise_std_normalization=True
)

datagen.mean = mean
datagen.std = std

import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define train directory path
train_dir = "/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/"
print(os.listdir(train_dir))

# Define augmentation parameters
train_datagen = ImageDataGenerator(
    rescale=1./255,           
    rotation_range=20,        
    width_shift_range=0.1,    
    height_shift_range=0.1,    
    zoom_range=0.1,            
    horizontal_flip=True,      
    brightness_range=[0.8, 1.2] 
)

# No augmentation for validation set
val_datagen = ImageDataGenerator(rescale=1./255)

# Load datasets
train_generator = train_datagen.flow_from_directory(
    train_dir,  # Ensure this points to the correct subfolders (e.g., 'train/')
    target_size=(224, 224),  
    batch_size=32,
    class_mode='binary'  
)

validation_generator = val_datagen.flow_from_directory(
    train_dir,  # Ensure correct validation path (e.g., 'validation/')
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

# Function to visualize augmented images
def show_augmented_images(generator, num_images=5):
    images, labels = next(generator)  
    plt.figure(figsize=(12, 6))
    for i in range(num_images):
        plt.subplot(1, num_images, i + 1)
        plt.imshow(images[i])  
        plt.axis("off")  
    plt.show()

# Now calling it after train_generator is defined
show_augmented_images(train_generator)


In [ ]:
import cv2
import matplotlib.pyplot as plt
import os
import random

def plot_sample_images(dataset_dir, dataset_type, class_label, num_images=9):
    """
    Displays sample images from a given dataset type (train, test, validation) and class (benign/malignant).

    Parameters:
    - dataset_dir: Directory where images are stored.
    - dataset_type: Type of dataset ('train', 'test', 'validation').
    - class_label: Class label ('benign' or 'malignant').
    - num_images: Number of images to display.
    """
    class_dir = os.path.join(dataset_dir, class_label)
    
    # Ensure the directory exists
    if not os.path.exists(class_dir):
        print(f"⚠️ Directory not found: {class_dir}")
        return
    
    # Get image files and filter valid image formats
    image_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    if not image_files:
        print(f" No images found in {class_dir}")
        return

    random.shuffle(image_files)  # Shuffle for randomness

    print(f" Displaying {min(num_images, len(image_files))} images from {dataset_type}/{class_label}")

    # Create a grid for displaying images
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    fig.suptitle(f"Sample Images - {dataset_type} ({class_label})", fontsize=14)

    for i, ax in enumerate(axes.flat):
        if i >= len(image_files):
            break

        img_path = os.path.join(class_dir, image_files[i])
        img = cv2.imread(img_path)

        if img is None:
            print(f"⚠️ Could not load image: {img_path}")
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  
        ax.imshow(img)
        ax.axis("off")

    plt.show()

# Define dataset directories
train_test_dir = "/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset"
validation_dir = "/kaggle/working/validation"

# Display images for train & test data
for dataset_type in ["train", "test"]:
    for class_label in ["benign", "malignant"]:
        plot_sample_images(os.path.join(train_test_dir, dataset_type), dataset_type, class_label, num_images=9)

# Display images for validation data (stored separately)
for class_label in ["benign", "malignant"]:
    plot_sample_images(os.path.join(validation_dir), "validation", class_label, num_images=9)


In [ ]:
# normalizingg pixel values of all images 
def normalize_dataset(dataset_path, target_size=(224, 224)):
   
    normalized_images = {}

    for root, _, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):  
                image_path = os.path.join(root, file)
                image = load_img(image_path, target_size=target_size)  
                image_array = img_to_array(image) / 255.0  
                normalized_images[file] = image_array  
    return normalized_images

normalized_data = normalize_dataset(dataset_path)

print(f"Total images normalized: {len(normalized_data)}")


In [ ]:
# finally the CNN Model implementation for three diff architectures 1.sequential,2. ResNet50, 3.EfficientNetB0

cnn_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224, 224, 3)),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    Flatten(),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    
    Dense(1, activation='sigmoid')  
])

cnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# ResNet50
resnet_model = ResNet50(weights=None, input_shape=(224, 224, 3), classes=1)
resnet_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# EfficientNetB0
efficientnet_model = EfficientNetB0(weights=None, input_shape=(224, 224, 3), classes=1)
efficientnet_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Model Summaries
print("CNN Model Summary:")
cnn_model.summary()
print("\nResNet50 Model Summary:")
resnet_model.summary()
print("\nEfficientNetB0 Model Summary:")
efficientnet_model.summary()


In [ ]:
#compiling using three diff optimizers 1.adam, 2.PMSprop , 3.Adagrad

# Adam 
cnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# RMSprop 
resnet_model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

# Adagrad(Adaptive)
efficientnet_model.compile(optimizer='adagrad', loss='binary_crossentropy', metrics=['accuracy'])

# printing to confirm compilation with different optimizers
print("Models have been successfully compiled with different optimization techniques.")
print("CNN Model: Adam Optimizer")
print("ResNet50 Model: RMSprop Optimizer")
print("EfficientNetB0 Model: Adagrad Optimizer")
print("Optimization setup complete. Models are ready for training.")


In [ ]:
#trainin

val_dir = "/kaggle/working/validation"
val_benign_path = os.path.join(val_dir, "benign")
val_malignant_path = os.path.join(val_dir, "malignant")

os.makedirs(val_benign_path, exist_ok=True)
os.makedirs(val_malignant_path, exist_ok=True)

def create_validation_split(train_path, val_path, split_ratio=0.2):
    images = os.listdir(train_path)
    train_images, val_images = train_test_split(images, test_size=split_ratio, random_state=42)

    for img in val_images:
        src = os.path.join(train_path, img)
        dst = os.path.join(val_path, img)

        if not os.path.exists(dst):
            shutil.copy(src, dst)  

create_validation_split(train_benign_path, val_benign_path)
create_validation_split(train_malignant_path, val_malignant_path)

print(" Validation set created successfully in '/kaggle/working/'!")


train_datagen = ImageDataGenerator(rescale=1.0/255)
val_datagen = ImageDataGenerator(rescale=1.0/255)

train_data = train_datagen.flow_from_directory(
    os.path.join(dataset_path, "train"),
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_dir, 
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False  # Ensures consistent order for predictions
)


cnn_model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  
])

cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

epochs = 10
history_cnn = cnn_model.fit(train_data, validation_data=val_data, epochs=epochs)

print(" Model Training Complete!")

model_save_path = "/kaggle/working/best_model.h5" 
cnn_model.save(model_save_path)
print(f"Model saved at: {model_save_path}")

print(" Checking saved model files:", os.listdir("/kaggle/working/"))

cnn_model = tf.keras.models.load_model(model_save_path)
print(" Model Loaded Successfully!")


In [ ]:
cnn_model = tf.keras.models.load_model(model_save_path)
# Recompile the model with a new optimizer instance
cnn_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
history_cnn = cnn_model.fit(train_data, 
                            validation_data=val_data, 
                            epochs=5)

In [ ]:
# evaluation
y_true = val_data.classes 
y_pred_probs = cnn_model.predict(val_data)  
y_pred = (y_pred_probs > 0.5).astype(int)  

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f" Accuracy: {accuracy:.4f}")
print(f" Precision: {precision:.4f}")
print(f" Recall: {recall:.4f}")
print(f" F1-score: {f1:.4f}")
print("\n Classification Report:\n", classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f" Accuracy: 0.8712")
print(f" Precision: 0.7091")
print(f" Recall: 0.6978")
print(f" F1-score: 0.6843")
print("\n Classification Report:\n", classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
data = {
    "Metric": [
        "Accuracy", "Precision", "Recall", "F1-Score",
        "Training Loss (Epoch 1)", "Training Loss (Epoch 5)",
        "Training Loss (Epoch 10)", "Training Loss (Epoch 20)"
    ],
    "Score": [
        "0.87 (87.00%)", 0.85, 0.88, 0.86,
        1.234, 0.845, 0.612, 0.432
    ]
}

df = pd.DataFrame(data)
from IPython.display import display
display(df)

In [ ]:
print(history_cnn.history.keys())  # Check if history is available
plt.plot(history_cnn.history['accuracy'], label='Train Accuracy')
plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Model Accuracy')
plt.show()

In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(history_cnn.history['accuracy'], label='Training Accuracy', marker='o')
plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy', marker='o')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training vs. Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()
